In [2]:
import time
import matplotlib.pyplot as plt
import numpy as np
import os

### Exercice 1 – Recuit simulé pour le problème du sac à dos 0/1

On vous demande dans cet exercice de programmer un algorithme de recuit simulé pour résoudre le problème
du sac à dos 0/1. Vous testerez votre algorithme sur les données fournies lors des séances précédentes. Vous
comparerez statistiquement (donc en faisant de multiples run de votre algorithme, ce qui est indispensable
quand l’aléatoire intervient) les résultats de votre algorithme avec ceux obtenus par une stratégie déterministe,
consistant à accepter uniquement les solutions améliorantes. Vous pourrez également :

- comparer les solutions obtenues avec les solutions optimales.
- analyser l’impact des différents paramètres de l’algorithme.

Pour générer un voisin à partir d’une solution initiale, plusieurs options pourront être imaginées. Une pos-
sibilité consiste à :

1. Tirer aléatoirement un objet non présent dans le sac
2. Ajouter cet objet
3. Tant que le poids du sac dépasse la capacité maximale :

(a) Tirer aléatoirement un objet présent dans le sac

(b) Supprimer cet objet

In [ ]:

def initial_solution(weights, values, wmax):
    W = int(wmax) 
    n = len(weights) 
    selected = np.zeros(n, dtype=int)
    result = 0

    while W >= 0 and np.any(selected == 0):  # Garante que há itens para adicionar
        i = np.random.randint(n)
        if selected[i] == 0 and W >= weights[i]:  # Apenas adiciona se houver espaço
            selected[i] = 1
            W -= weights[i]
            result += values[i]

    return result, W, selected

In [4]:
def Knapsack_DA(weights, values, wmax):
    result, W, selected = initial_solution(weights, values, wmax)
    n = len(weights)

    Best_list = selected.copy()
    Best_result = result
    Best_W = W

    print(f"{Best_list}, {Best_W}, {Best_result} (Initial)\n")

    for iteration in range(1000): 
        voisin_list = Best_list.copy()
        voisin_result = Best_result
        voisin_W = Best_W
        
        i = np.random.randint(n)
        if voisin_list[i] == 0 and voisin_W >= weights[i]:  # Apenas adiciona se houver espaço
            voisin_list[i] = 1
            voisin_W -= weights[i]
            voisin_result += values[i]

            print(f"{voisin_list}, {voisin_W}, {voisin_result} (add{i}, {weights[i]}, {values[i]})")

            while voisin_W < 0:
                j = np.random.randint(n)
                if voisin_list[j] == 1:
                    voisin_list[j] = 0
                    voisin_W += weights[j]
                    voisin_result -= values[j]
                    print(f"{voisin_list}, {voisin_W}, {voisin_result} (remove{j}, {weights[j]}, {values[j]})")

            if voisin_result >= Best_result:
                Best_list = voisin_list.copy()
                Best_W = voisin_W 
                Best_result = voisin_result

                print(f"{Best_list}, {Best_W}, {Best_result} (New Best)\n")

    return Best_result, Best_list
    

In [11]:
import numpy as np

def initial_solution(weights, values, wmax):
    """Gera uma solução inicial viável para o problema da mochila."""
    n = len(weights)
    selected = np.zeros(n, dtype=int)
    W = wmax
    result = 0

    indices = np.random.permutation(n)  # Embaralha os índices para melhor aleatoriedade

    for i in indices:
        if W >= weights[i]:
            selected[i] = 1
            W -= weights[i]
            result += values[i]

    return selected, result, W

def knapsack_value(selected, weights, values):
    """Calcula o valor total e o peso de uma configuração."""
    value = np.dot(selected, values)
    weight = np.dot(selected, weights)
    return value, weight

def get_neighbor(selected, weights, values, wmax):
    """Gera uma solução vizinha ao modificar um item da configuração atual."""
    neighbor = selected.copy()
    n = len(weights)

    i = np.random.randint(n)
    neighbor[i] = 1 - neighbor[i]  # Alterna entre 0 e 1 (adiciona ou remove item)

    new_value, new_weight = knapsack_value(neighbor, weights, values)

    # Se ultrapassar o peso máximo, reverte a mudança
    if new_weight > wmax:
        neighbor[i] = selected[i]  # Desfaz a mudança
        new_value, new_weight = knapsack_value(neighbor, weights, values)

    return neighbor, new_value, new_weight

def simulated_annealing_knapsack(weights, values, wmax, T0=100, coeff=0.99, iter_step=100, stop_threshold=0.001):
    """Resolve o problema da mochila 0/1 usando Simulated Annealing."""
    
    # Inicialização
    current_solution, current_value, current_weight = initial_solution(weights, values, wmax)
    best_solution = current_solution.copy()
    best_value = current_value
    T = T0

    # Loop principal
    while T > stop_threshold:
        nb_moves = 0
        for i in range(iter_step):
            # Gera vizinho
            new_solution, new_value, new_weight = get_neighbor(current_solution, weights, values, wmax)
            
            # Calcula delta em relação à solução ATUAL
            delta = new_value - current_value

            # Critério de Metropolis
            if delta > 0 or np.random.rand() < np.exp(delta / T):
                current_solution = new_solution.copy()
                current_value = new_value
                current_weight = new_weight
                nb_moves += 1

                # Atualiza melhor solução encontrada
                if current_value > best_value:
                    best_solution = current_solution.copy()
                    best_value = current_value

        # Atualiza a temperatura
        acceptance_rate = nb_moves / iter_step if iter_step > 0 else 0
        if acceptance_rate < stop_threshold:
            break  # Critério de parada se a aceitação for muito baixa
        T *= coeff

    return best_solution, best_value


In [5]:
path = "./instances_01_KP/low-dimensional/f10_l-d_kp_20_879" 
dataset = np.loadtxt(path)

n = int(dataset[0][0])
wmax = dataset[0][1]
itens = dataset[1:]

weights = dataset[1:,1]
values = dataset[1:,0]

print(weights)
print(values)

[84. 83. 43.  4. 44.  6. 82. 92. 25. 83. 56. 18. 58. 14. 48. 70. 96. 32.
 68. 92.]
[91. 72. 90. 46. 55.  8. 35. 75. 61. 15. 77. 40. 63. 75. 29. 75. 17. 78.
 40. 44.]


In [12]:
for i in range(4):
    best_solution, best_value = simulated_annealing_knapsack(weights, values, wmax)
    print("Melhor valor encontrado:", best_value)
    print("Itens selecionados:", best_solution)

Melhor valor encontrado: 1025.0
Itens selecionados: [1 1 1 1 1 1 1 1 1 0 1 1 1 1 0 1 0 1 1 1]
Melhor valor encontrado: 1025.0
Itens selecionados: [1 1 1 1 1 1 1 1 1 0 1 1 1 1 0 1 0 1 1 1]
Melhor valor encontrado: 1025.0
Itens selecionados: [1 1 1 1 1 1 1 1 1 0 1 1 1 1 0 1 0 1 1 1]
Melhor valor encontrado: 1025.0
Itens selecionados: [1 1 1 1 1 1 1 1 1 0 1 1 1 1 0 1 0 1 1 1]
